# 05 — Edge AI and TensorFlow Lite Deployment
## Last notebook: take a trained model towards a device

This notebook focuses on exporting a model, verifying the converted artifact and understanding what changes when inference moves from a laptop to an edge device.

Prerequisite: complete Notebook 02 and have `cifar10_cnn.keras` available.

## 1. Load the trained CNN

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

model = tf.keras.models.load_model("cifar10_cnn.keras")
model.summary()

## 2. Convert Keras model to TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

output_path = Path("cifar10_cnn.tflite")
output_path.write_bytes(tflite_model)

print("Saved:", output_path)
print("Model size (KB):", round(len(tflite_model) / 1024, 2))

## 3. Inspect input and output tensors

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(output_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details)
print("Output details:", output_details)

## 4. Compare one Keras and TFLite prediction

In [ ]:
(_, _), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
sample = x_test[0:1].astype("float32") / 255.0

keras_scores = model.predict(sample, verbose=0)

interpreter.set_tensor(input_details[0]["index"], sample.astype(input_details[0]["dtype"]))
interpreter.invoke()
lite_scores = interpreter.get_tensor(output_details[0]["index"])

print("Keras predicted class:", int(np.argmax(keras_scores[0])))
print("TFLite predicted class:", int(np.argmax(lite_scores[0])))
print("Maximum absolute difference:", float(np.max(np.abs(keras_scores - lite_scores))))

## 5. Inspect the model with Netron

Open https://netron.app and load `cifar10_cnn.tflite`.

Inspect:
- Input tensor shape.
- Convolution operations and filter dimensions.
- Feature-map shape changes.
- Dense/output operations and 10 output scores.
- Weights and biases.

Netron visualizes model structure and metadata; it does not train the model. Converted graphs may fuse operations or contain conversion-specific operators.

## 6. What makes deployment an Edge AI problem?

A Raspberry Pi or other edge device has limited compute, memory, power and thermal headroom. Before deployment, measure:
- Model file size and RAM use.
- Inference latency and throughput.
- Power draw and temperature.
- Accuracy after conversion/quantization.
- Input preprocessing and output handling.

Quantization can represent weights and activations with lower precision (for example, int8 rather than float32), often reducing model size and memory traffic. It may change accuracy, so compare results rather than assuming it is always better.

## 7. Final distinction
- Training: update weights using data and gradients.
- Inference: use fixed learned weights to predict.
- Conversion: transform the model into a deployment format.
- Deployment: integrate and test the model on the target device.